# **Random Forest**



**Research Paper Links**

Base — Random Forest (Breiman, 2001): https://link.springer.com/article/10.1023/A:1010933404324



Variant 1 — Mondrian Forests (Lakshminarayanan et al., 2014): https://arxiv.org/abs/1406.2673







Variant 2 — Deep Forest (Zhou & Feng, 2019): https://academic.oup.com/nsr/article/6/1/74/5DetailsPage

# Random Forest (Base)

In [1]:
# Arabic Sentiment Analysis — Random Forest (Base)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_clean.csv"
VAL_PATH      = "/content/val_clean.csv"
TEST_PATH     = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

# Fit TF-IDF ONLY on training data
tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

# Combine with emoji features (keep sparse)
def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Random Forest
print("=" * 60)
print("  🌲 Training: Random Forest (Base)")
print("=" * 60)

clf = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = None,       # fully grown trees
    min_samples_leaf = 2,          # slight regularisation
    class_weight     = "balanced", # handles label imbalance
    n_jobs           = -1,
    random_state     = RANDOM_STATE,
)

clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Random Forest  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 10,264
Final feature shape (train) : (5067, 10266)

Classes : [0 1]  →  [0, 1]

  🌲 Training: Random Forest (Base)

────────────────────────────────────────────────────────────
  Random Forest  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7643
  Precision : 0.7644
  Recall    : 0.7643
  F1-Score  : 0.7642

  Classification Report:

              precision    recall  f1-score   support

           0       0.76      0.78      0.77       543
           1       0.77      0.75      0.76       543

    accuracy                           0.76      1086
   macro avg       0.76      0.76      0.76      1086
weighted avg       0.76      0.76      0.76      1086

  Confusion Matrix:
[[421 122]
 [134 409]]


──────────────────────────────────────────────────────────